In [1]:
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import bronze_inventory

settings = Settings.from_environment()
for path, size in bronze_inventory(settings):
    print(path, size)

bronze/source=google_qec/google-surface-code-curated.zip 14638673
bronze/source=qasmbench/qasmbench-qec.zip 144172
bronze/source=qec_syndromes/syndromes_dataset.zip 358017


In [2]:
import io
import zipfile

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()
client = minio_client(settings)

bronze_zips = [
    "bronze/source=google_qec/google-surface-code-curated.zip",
    "bronze/source=qasmbench/qasmbench-qec.zip",
    "bronze/source=qec_syndromes/syndromes_dataset.zip",
]

for key in bronze_zips:
    print(f"\n=== {key} ===")
    response = client.get_object(settings.s3_bucket, key)
    data = response.read()
    response.close()
    response.release_conn()

    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for info in z.infolist():
            print(f"  {info.filename}  ({info.file_size} bytes)")


=== bronze/source=google_qec/google-surface-code-curated.zip ===
  README.txt  (15656 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem  (172277 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim  (18773 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim  (124790 bytes)
  surface_code_bX_d3_r25_center_3_5/detection_events.b8  (1250000 bytes)
  surface_code_bX_d3_r25_center_3_5/layout.svg  (17685 bytes)
  surface_code_bX_d3_r25_center_3_5/measurements.b8  (1350000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01  (100000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_belief_matching.01  (100000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_correlated_matching.01  (100000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_pymatching.01  (100000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_tensor_network_contraction.01  (100000 bytes)
  surface_code_bX_

In [3]:
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()          # loads your MinIO/Postgres credentials from .env
client = minio_client(settings)                  # creates a connection to MinIO

key = "bronze/source=qec_syndromes/syndromes_dataset.zip"
response = client.get_object(settings.s3_bucket, key)  # download the zip bytes (not extracted to disk)
data = response.read()
response.close()
response.release_conn()

with zipfile.ZipFile(io.BytesIO(data)) as z:      # open the zip bytes as if it were a file, in memory
    with z.open("d-3_pfr-0.000010_nb-10M.csv") as f:  # open just ONE file inside the zip, without extracting
        lines = f.read().decode().splitlines()   # read its text, split into individual lines

print("Header:", lines[0])          # column names
print("First row:", lines[1])       # the first actual data row
print("Total rows (incl header):", len(lines))

Header: labels,syndromes,quantity
First row: 0,"((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))",9987291
Total rows (incl header): 69


In [4]:
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()
client = minio_client(settings)

key = "bronze/source=google_qec/google-surface-code-curated.zip"
response = client.get_object(settings.s3_bucket, key)
data = response.read()
response.close()
response.release_conn()

with zipfile.ZipFile(io.BytesIO(data)) as z:
    # properties.yml is tiny (278 bytes) — likely has qubit/round/stabilizer counts we need
    with z.open("surface_code_bX_d3_r25_center_3_5/properties.yml") as f:
        print(f.read().decode())

type: surface_code_memory_experiment
basis: X
rounds: 25
distance: 3
data_qubits: 9
measure_qubits: 8
shots: 50000
center_data_qubit_row: 3
center_data_qubit_col: 5
circuit_measurements: 209
circuit_sweep_bits: 9
circuit_detectors: 200
circuit_observables: 1
circuit_qubits: 17



In [5]:
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.formats import iter_b8_records, b8_record_bytes

settings = Settings.from_environment()
client = minio_client(settings)

key = "bronze/source=google_qec/google-surface-code-curated.zip"
response = client.get_object(settings.s3_bucket, key)
data = response.read()
response.close()
response.release_conn()

with zipfile.ZipFile(io.BytesIO(data)) as z:
    with z.open("surface_code_bX_d3_r25_center_3_5/detection_events.b8") as f:
        raw = f.read()

bits_per_record = 200          # from properties.yml: circuit_detectors
record_size = b8_record_bytes(bits_per_record)   # bytes needed per record
print("File size:", len(raw), "bytes")
print("Bytes per record:", record_size)
print("Expected records:", len(raw) / record_size)   # should be a clean whole number = 50000

records = list(iter_b8_records(raw, bits_per_record=bits_per_record))
print("Actual record count:", len(records))
print("First record (200 bits):", records[0])
print("Sum of 1-bits in first record:", sum(records[0]))  # sanity: how many detectors fired


File size: 1250000 bytes
Bytes per record: 25
Expected records: 50000.0
Actual record count: 50000
First record (200 bits): (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0)
Sum of 1-bits in first record: 27


In [6]:
from quantum_lake_student.formats import parse_01_records
key = "bronze/source=google_qec/google-surface-code-curated.zip"
response = client.get_object(settings.s3_bucket, key)
data = response.read()
response.close()
response.release_conn()

with zipfile.ZipFile(io.BytesIO(data)) as z:
    with z.open("surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01") as f:
        raw = f.read()

records = parse_01_records(raw)      # course helper: parses one 0/1 value per line
print("Total records:", len(records))       # should be 50000, matching 'shots' in properties.yml
print("First 20 values:", records[:20])
print("Unique values:", set(records))       # should only be {0, 1} - confirms no corrupted lines
print("Fraction of logical errors:", sum(records) / len(records))  # how often the logical qubit actually flipped


Total records: 50000
First 20 values: [1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
Unique values: {0, 1}
Fraction of logical errors: 0.49034
